#  PD Metabolomics Annotator
Fetches **PubChem CID**, **KEGG ID**, **HMDB ID**, **molecular formula**, **InChIKey**, and **SMILES**
for every metabolite

**Lookup order per metabolite:**
1. PubChem — via InChIKey → name → HMDB cross-reference
2. KEGG — via name search and additional look up
3. HMDB — via existing ID fetch → name search fallback

**Run cells top to bottom.** The cache saves progress so you can resume if interrupted.


## 1 · Install dependencies

In [ ]:
!pip install -q requests pandas openpyxl tqdm
print("✅ Dependencies ready")


✅ Dependencies ready


## 2 · Mount Google Drive
Your input file and all outputs (annotated Excel + cache) will be saved to Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive mounted at /content/drive")


Mounted at /content/drive
✅ Drive mounted at /content/drive


## 3 · Configuration
Set the path to your input file and where outputs should go.


In [ ]:
import os
from google.colab import files

uploaded = files.upload()
INPUT_FILE = next(iter(uploaded))  # extracts filename from the dict

OUTPUT_FILE = "/content/drive/MyDrive/PD_metabolomics_annotated_new.xlsx"
CACHE_FILE  = "/content/drive/MyDrive/annotation_cache.json"

DELAY      = 0.35
TEST_LIMIT = None

print(f"✅ Input : {INPUT_FILE}")
print(f"   Output: {OUTPUT_FILE}")
print(f"   Cache : {CACHE_FILE}")

Saving kegg.xlsx to kegg.xlsx
✅ Input : kegg.xlsx
   Output: /content/drive/MyDrive/PD_metabolomics_annotated_new.xlsx
   Cache : /content/drive/MyDrive/annotation_cache.json


## 4 · API functions  *(run once, no output expected)*

In [ ]:
import time, json, re, logging
import requests

logging.basicConfig(level=logging.WARNING)   # suppress verbose logs in Colab

SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "PD-Metabolomics-Annotator-Colab/1.0"})

PUBCHEM_BASE = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"
KEGG_BASE    = "https://rest.kegg.jp"
HMDB_BASE    = "https://hmdb.ca"
_PC_PROPS    = "MolecularFormula,InChIKey,IsomericSMILES"


def _get(url, params=None, timeout=20):
    for attempt in range(3):
        try:
            r = SESSION.get(url, params=params, timeout=timeout)
            if r.status_code == 200:
                return r
            if r.status_code == 404:
                return None
            if r.status_code == 429:
                wait = 10 * (attempt + 1)
                print(f"  ⚠️  Rate-limited — sleeping {wait}s …")
                time.sleep(wait)
            else:
                time.sleep(1)
        except requests.RequestException:
            time.sleep(2)
    return None


# ── PubChem ───────────────────────────────────────────────────────────────────

def _pc_parse(r):
    try:
        p = r.json()["PropertyTable"]["Properties"][0]
        return {
            "pubchem_cid": str(p.get("CID", "")),
            "formula":     p.get("MolecularFormula", ""),
            "inchikey":    p.get("InChIKey", ""),
            "smiles":      p.get("IsomericSMILES", ""),
        }
    except Exception:
        return {}

def pubchem_by_inchikey(ik):
    ik = ik.replace("InChI=1S/", "").replace("InChIKey=", "").strip()
    if not ik or len(ik) < 14:
        return {}
    r = _get(f"{PUBCHEM_BASE}/compound/inchikey/{requests.utils.quote(ik)}/property/{_PC_PROPS}/JSON")
    return _pc_parse(r) if r else {}

def pubchem_by_name(name):
    r = _get(f"{PUBCHEM_BASE}/compound/name/{requests.utils.quote(name)}/property/{_PC_PROPS}/JSON")
    return _pc_parse(r) if r else {}

def pubchem_by_hmdb(hmdb_id):
    hid = hmdb_id.upper().strip()
    hid_short = re.sub(r"HMDB0+(\d{5})$", r"HMDB\1", hid)
    for candidate in set([hid, hid_short]):
        r = _get(f"{PUBCHEM_BASE}/compound/xref/RegistryID/{requests.utils.quote(candidate)}/property/{_PC_PROPS}/JSON")
        if r:
            result = _pc_parse(r)
            if result:
                return result
    return {}


# ── KEGG ──────────────────────────────────────────────────────────────────────
# ============================================================
# Map metabolites to KEGG Compound IDs
# Run this in Google Colab as separate cells (split at the "# %%" markers)
# ============================================================

# %% Cell 1 — install & imports
!pip install -q requests pandas tqdm

import requests
import pandas as pd
import time
import json
from tqdm.notebook import tqdm

# %% Cell 2 — upload your file
from google.colab import files
uploaded = files.upload()
input_filename = list(uploaded.keys())[0]
print("Uploaded:", input_filename)

# %% Cell 3 — load data
df = pd.read_csv(input_filename)
df.columns = [c.strip() for c in df.columns]
print(df.columns.tolist())
print(df.shape)
df.head()

# ---- EDIT THESE if your column names differ ----
COL_NAME    = "canonical_metabolite"
COL_INCHIKEY = "InChIKey"
COL_FORMULA = "formula"
COL_PUBCHEM = "PubChem CID"
# --------------------------------------------------

# %% Cell 4 — UniChem lookup function (primary method, keyed on InChIKey)
# UniChem (EBI) source ID 6 = KEGG Ligand/Compound
UNICHEM_URL = "https://www.ebi.ac.uk/unichem/api/v1/compounds"
session = requests.Session()

def unichem_kegg_from_inchikey(inchikey, retries=3, timeout=15):
    if not isinstance(inchikey, str) or len(inchikey.strip()) < 14:
        return None
    payload = {"type": "inchikey", "compound": inchikey.strip()}
    for attempt in range(retries):
        try:
            r = session.post(UNICHEM_URL, json=payload, timeout=timeout)
            if r.status_code == 404:
                return None
            r.raise_for_status()
            data = r.json()
            # Response shape: list of compound records, each with "sources": [{"shortName"/"name":..., "sourceID":..., "compoundId":...}, ...]
            # Be defensive about the exact key names since the API is fairly new.
            records = data if isinstance(data, list) else data.get("compounds", data.get("results", []))
            for rec in records:
                sources = rec.get("sources") or rec.get("Sources") or []
                for s in sources:
                    src_id = s.get("sourceID") or s.get("srcId") or s.get("src_id")
                    src_name = (s.get("name") or s.get("shortName") or "").lower()
                    if src_id == 6 or "kegg" in src_name:
                        cid = s.get("compoundId") or s.get("src_compound_id") or s.get("srcCompoundId")
                        if cid:
                            return cid
            return None
        except (requests.RequestException, json.JSONDecodeError, ValueError):
            time.sleep(1.5 * (attempt + 1))
    return None

# %% Cell 5 — run primary lookup over all rows (checkpointed, resumable)
CHECKPOINT_FILE = "kegg_lookup_checkpoint.csv"

if "kegg_id" not in df.columns:
    df["kegg_id"] = None
if "match_method" not in df.columns:
    df["match_method"] = None

try:
    ckpt = pd.read_csv(CHECKPOINT_FILE)
    df.loc[:, "kegg_id"] = ckpt["kegg_id"]
    df.loc[:, "match_method"] = ckpt["match_method"]
    print("Resumed from checkpoint.")
except FileNotFoundError:
    pass

todo_idx = df.index[df["kegg_id"].isna()]
print(f"{len(todo_idx)} rows left to look up")

for i, idx in enumerate(tqdm(todo_idx)):
    ik = df.at[idx, COL_INCHIKEY]
    kid = unichem_kegg_from_inchikey(ik)
    if kid:
        df.at[idx, "kegg_id"] = kid
        df.at[idx, "match_method"] = "unichem_inchikey"
    time.sleep(0.15)  # be polite to the API (~6-7 req/sec)
    if i % 100 == 0:
        df.to_csv(CHECKPOINT_FILE, index=False)

df.to_csv(CHECKPOINT_FILE, index=False)
print("Primary pass done.")
print("Matched via UniChem:", df["kegg_id"].notna().sum(), "/", len(df))

# %% Cell 6 — fallback for unmatched rows: KEGG "find by name", verified against formula
KEGG_FIND_URL = "https://rest.kegg.jp/find/compound/{}"
KEGG_GET_URL = "https://rest.kegg.jp/get/{}"

def kegg_find_candidates_by_name(name, timeout=15):
    """Returns list of (kegg_id, description) candidates from KEGG name search."""
    try:
        r = session.get(KEGG_FIND_URL.format(requests.utils.quote(name)), timeout=timeout)
        if r.status_code != 200 or not r.text.strip():
            return []
        out = []
        for line in r.text.strip().split("\n"):
            parts = line.split("\t")
            if len(parts) == 2 and parts[0].startswith("cpd:"):
                out.append((parts[0].replace("cpd:", ""), parts[1]))
        return out
    except requests.RequestException:
        return []

def kegg_formula_for(kegg_id, timeout=15):
    """Fetch a KEGG compound entry and pull its FORMULA line for verification."""
    try:
        r = session.get(KEGG_GET_URL.format(kegg_id), timeout=timeout)
        if r.status_code != 200:
            return None
        for line in r.text.split("\n"):
            if line.startswith("FORMULA"):
                return line.replace("FORMULA", "").strip()
        return None
    except requests.RequestException:
        return None

remaining_idx = df.index[df["kegg_id"].isna()]
print(f"{len(remaining_idx)} rows remain for name/formula fallback")

for idx in tqdm(remaining_idx):
    name = str(df.at[idx, COL_NAME])
    formula = str(df.at[idx, COL_FORMULA]).strip()
    candidates = kegg_find_candidates_by_name(name)
    time.sleep(0.2)
    if not candidates:
        continue
    # Verify by formula match to avoid false positives from fuzzy name search
    matched = None
    for kid, desc in candidates[:5]:
        f = kegg_formula_for(kid)
        time.sleep(0.2)
        if f and f == formula:
            matched = kid
            break
    if matched:
        df.at[idx, "kegg_id"] = matched
        df.at[idx, "match_method"] = "kegg_name_formula_verified"
    elif candidates:
        # No formula-verified hit; record the top name-only candidate for manual review
        df.at[idx, "kegg_id"] = candidates[0][0]
        df.at[idx, "match_method"] = "kegg_name_only_UNVERIFIED"

df.to_csv(CHECKPOINT_FILE, index=False)

# %% Cell 7 — summary & save final output
print("Final match summary:")
print(df["match_method"].value_counts(dropna=False))
print("Total matched:", df["kegg_id"].notna().sum(), "/", len(df))

output_filename = "metabolites_with_kegg_ids.csv"
df.to_csv(output_filename, index=False)
files.download(output_filename)
def kegg_by_name(name):
    r = _get(f"{KEGG_BASE}/find/compound/{requests.utils.quote(name)}")
    if r is None or not r.text.strip():
        return {}
    kegg_id = r.text.strip().splitlines()[0].split("\t")[0].replace("cpd:", "").strip()
    return kegg_fetch(kegg_id) if kegg_id else {}

def kegg_fetch(kegg_id):
    if not kegg_id:
        return {}
    r = _get(f"{KEGG_BASE}/get/{kegg_id}")
    if r is None:
        return {}
    result = {"kegg_id": kegg_id, "formula": "", "inchikey": ""}
    for line in r.text.splitlines():
        if line.startswith("FORMULA"):
            result["formula"] = line.replace("FORMULA", "").strip()
        elif "InChIKey=" in line:
            result["inchikey"] = line.split("InChIKey=")[-1].strip()
    return result



# ── HMDB ──────────────────────────────────────────────────────────────────────

def _parse_hmdb_xml(text):
    def tag(t):
        m = re.search(rf"<{t}[^>]*>([^<]+)</{t}>", text)
        return m.group(1).strip() if m else ""
    accession = tag("accession")
    hmdb_id   = accession if accession.startswith("HMDB") else ""
    if not hmdb_id:
        m = re.search(r"<accession>(HMDB\d+)</accession>", text)
        hmdb_id = m.group(1) if m else ""
    return {k: v for k, v in {
        "hmdb_id":  hmdb_id,
        "formula":  tag("chemical_formula"),
        "inchikey": tag("inchikey"),
        "smiles":   tag("smiles"),
    }.items() if v}

def hmdb_fetch(hmdb_id):
    if not hmdb_id or hmdb_id.lower() in ("nan", ""):
        return {}
    digits = re.sub(r"(?i)hmdb0*", "", hmdb_id.strip())
    for fmt in [f"HMDB{digits.zfill(7)}", f"HMDB{digits.zfill(5)}"]:
        r = _get(f"{HMDB_BASE}/metabolites/{fmt}.xml")
        if r and "<metabolite>" in r.text:
            result = _parse_hmdb_xml(r.text)
            if result:
                result.setdefault("hmdb_id", fmt)
                return result
    return {}

def hmdb_by_name(name):
    r = _get(f"{HMDB_BASE}/metabolites/search",
             params={"query": name, "search_type": "metabolites"})
    if r is None:
        return {}
    m = re.search(r"<accession>(HMDB\d+)</accession>", r.text)
    return hmdb_fetch(m.group(1)) if m else {}


# ── Name cleaner (improves API hit rate) ──────────────────────────────────────

def clean_name_for_lookup(name):
    """Strip asterisks, adduct annotations, and other noise before API lookup."""
    n = name.strip()
    n = re.sub(r"\*$", "", n)                        # trailing asterisk
    n = re.sub(r"\s*\([^)]*\+[^)]*\)", "", n)      # adducts like (+ H+), (+ Na+)
    n = re.sub(r"\s*-H2O$", "", n, flags=re.I)       # -H2O suffix
    n = re.sub(r"\s+\([12]\)$", "", n)              # disambiguation (1) (2)
    n = re.sub(r"\s*\*$", "", n)
    return n.strip()


# ── Core annotator ────────────────────────────────────────────────────────────

def annotate_metabolite(row, delay=0.35):
    name     = clean_name_for_lookup(str(row.get("metabolite", "")))
    raw_name = str(row.get("metabolite", "")).strip()
    formula  = str(row.get("formula",   "")).strip()
    inchikey = str(row.get("inchikey",  "")).strip()
    hmdb_id  = str(row.get("hmdb_id",  "")).strip()
    smiles   = str(row.get("smiles",   "")).strip()

    def clean(v): return "" if v.lower() in ("nan", "none", "n/a") else v
    name, formula, inchikey, hmdb_id, smiles = map(clean, [name, formula, inchikey, hmdb_id, smiles])

    pc, kg, hm = {}, {}, {}

    # 1. PubChem via InChIKey
    if inchikey:
        pc = pubchem_by_inchikey(inchikey); time.sleep(delay)
    # 2. PubChem via cleaned name
    if not pc.get("pubchem_cid") and name:
        pc = pubchem_by_name(name); time.sleep(delay)
    # 2b. PubChem via raw name if cleaned name failed
    if not pc.get("pubchem_cid") and raw_name != name:
        pc = pubchem_by_name(raw_name); time.sleep(delay)
    # 3. PubChem via HMDB xref
    if not pc.get("pubchem_cid") and hmdb_id:
        pc = pubchem_by_hmdb(hmdb_id); time.sleep(delay)
    # 4. KEGG via name
    if name:
        kg = kegg_by_name(name); time.sleep(delay)
    # 5. HMDB fetch by existing ID
    if hmdb_id:
        hm = hmdb_fetch(hmdb_id); time.sleep(delay)
    # 6. HMDB search by name (only if no ID yet)
    if not hm.get("hmdb_id") and not hmdb_id and name:
        hm = hmdb_by_name(name); time.sleep(delay)

    return {
        "pubchem_cid":     pc.get("pubchem_cid", ""),
        "kegg_id":         kg.get("kegg_id",     ""),
        "hmdb_id_filled":  hmdb_id  or hm.get("hmdb_id",  ""),
        "formula_filled":  formula  or pc.get("formula",  "") or kg.get("formula",  "") or hm.get("formula",  ""),
        "inchikey_filled": inchikey or pc.get("inchikey", "") or kg.get("inchikey", "") or hm.get("inchikey", ""),
        "smiles_filled":   smiles   or pc.get("smiles",   "") or hm.get("smiles",   ""),
    }

print("✅ API functions loaded")


✅ API functions loaded


## 5 · Load the unified dataset

In [ ]:

import pandas as pd

# Read first sheet automatically — no need to specify a sheet name
df = pd.read_excel(INPUT_FILE, sheet_name=0)

if TEST_LIMIT:
    df = df.iloc[:TEST_LIMIT].copy()
    print(f"⚠️  Test mode: using first {TEST_LIMIT} rows only")


# Detect columns flexibly
met_col     = next((c for c in df.columns if "metabolite" in c.lower()), None)
formula_col = next((c for c in df.columns if "formula"   in c.lower() and "fill" not in c.lower()), None)
ik_col      = next((c for c in df.columns if "inchikey"  in c.lower() and "fill" not in c.lower()), None)
hmdb_col    = next((c for c in df.columns if "hmdb"      in c.lower() and "fill" not in c.lower()), None)
smiles_col  = next((c for c in df.columns if "smiles"    in c.lower() and "fill" not in c.lower()), None)

assert met_col, "❌ Could not find a metabolite name column — check SHEET_NAME"

def first_nonempty(series):
    s = series.dropna().astype(str)
    s = s[~s.str.strip().str.lower().isin(["nan", "none", "n/a", ""])]
    return s.iloc[0].strip() if len(s) else ""

unique_names = (
    df[met_col].dropna().astype(str).str.strip()
    .pipe(lambda s: s[~s.str.lower().isin(["nan", ""])])
    .unique()
)

print(f"✅ Loaded {len(df):,} rows  |  {len(unique_names):,} unique metabolites")
print(f"   Columns → metabolite: '{met_col}' | formula: '{formula_col}' | inchikey: '{ik_col}' | hmdb: '{hmdb_col}'")


✅ Loaded 776 rows  |  776 unique metabolites
   Columns → metabolite: 'metabolite' | formula: 'formula' | inchikey: 'InChIKey' | hmdb: 'HMDB ID'


## 6 · Run annotation
This is the main loop. It will take a while for 1,600+ metabolites (~30–45 min at 0.35 s delay).
**Progress is saved to the cache file every 50 metabolites** — if the session disconnects,
re-run this cell and it will pick up where it left off.


In [ ]:
from tqdm.notebook import tqdm

# Load cache (supports resuming)
def load_cache(path):
    try:
        with open(path) as f: return json.load(f)
    except Exception: return {}

def save_cache(cache, path):
    with open(path, "w") as f: json.dump(cache, f, indent=2)

cache = load_cache(CACHE_FILE)
print(f"📂 Cache loaded: {len(cache)} entries already annotated")

met_annotations = {}
skipped = 0

for name in tqdm(unique_names, desc="Annotating metabolites"):
    if name in cache:
        met_annotations[name] = cache[name]
        skipped += 1
        continue

    mask = df[met_col].astype(str).str.strip() == name
    row_data = {
        "metabolite": name,
        "formula":  first_nonempty(df.loc[mask, formula_col]) if formula_col else "",
        "inchikey": first_nonempty(df.loc[mask, ik_col])      if ik_col     else "",
        "hmdb_id":  first_nonempty(df.loc[mask, hmdb_col])    if hmdb_col   else "",
        "smiles":   first_nonempty(df.loc[mask, smiles_col])  if smiles_col else "",
    }

    ann = annotate_metabolite(row_data, delay=DELAY)
    met_annotations[name] = ann
    cache[name] = ann

    # Save cache every 50 new annotations
    new_count = sum(1 for n in unique_names if n in cache) - skipped
    if new_count % 50 == 0 and new_count > 0:
        save_cache(cache, CACHE_FILE)

save_cache(cache, CACHE_FILE)
print(f"\n✅ Done!  {skipped} from cache  |  {len(unique_names) - skipped} newly fetched")


📂 Cache loaded: 0 entries already annotated


Annotating metabolites:   0%|          | 0/776 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 7 · Merge annotations into dataframe and save

In [ ]:
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font
from openpyxl.utils import get_column_letter

ann_output_cols = [
    ("pubchem_cid",     "PubChem CID"),
    ("kegg_id",         "KEGG ID"),
    ("hmdb_id_filled",  "HMDB ID (filled)"),
    ("formula_filled",  "Formula (filled)"),
    ("inchikey_filled", "InChIKey (filled)"),
    ("smiles_filled",   "SMILES (filled)"),
]

name_series = df[met_col].astype(str).str.strip()
for ann_key, col_label in ann_output_cols:
    df[col_label] = name_series.map(
        lambda n, k=ann_key: met_annotations.get(n, {}).get(k, "")
    )

df.to_excel(OUTPUT_FILE, index=False, sheet_name="Annotated_Metabolites")

# Formatting
wb = load_workbook(OUTPUT_FILE)
ws = wb.active

hdr_fill = PatternFill("solid", start_color="1F4E79", end_color="1F4E79")
hdr_font = Font(bold=True, color="FFFFFF", name="Arial", size=10)
ann_fill = PatternFill("solid", start_color="E2EFDA", end_color="E2EFDA")
alt_fill = PatternFill("solid", start_color="EBF3FB", end_color="EBF3FB")

new_labels     = {lbl for _, lbl in ann_output_cols}
header_vals    = [ws.cell(1, c).value for c in range(1, ws.max_column + 1)]
ann_indices    = {i + 1 for i, h in enumerate(header_vals) if h in new_labels}

for ci in range(1, ws.max_column + 1):
    c = ws.cell(1, ci)
    c.font = hdr_font
    c.fill = hdr_fill
    ws.column_dimensions[get_column_letter(ci)].width = 20

for ci, h in enumerate(header_vals, 1):
    if h and "metabolite" in str(h).lower():
        ws.column_dimensions[get_column_letter(ci)].width = 38
    elif h and any(x in str(h).lower() for x in ["inchikey", "smiles", "formula"]):
        ws.column_dimensions[get_column_letter(ci)].width = 32

for ri in range(2, ws.max_row + 1):
    row_fill = alt_fill if ri % 2 == 0 else None
    for ci in range(1, ws.max_column + 1):
        cell = ws.cell(ri, ci)
        cell.font = Font(name="Arial", size=9)
        if ci in ann_indices:
            cell.fill = ann_fill
        elif row_fill:
            cell.fill = row_fill

ws.freeze_panes = "A2"
ws.auto_filter.ref = ws.dimensions

# Summary sheet
ws2 = wb.create_sheet("Annotation_Summary")
filled = {lbl: int((df[lbl].astype(str).str.strip() != "").sum()) for _, lbl in ann_output_cols}
summary_rows = [
    ["Metric", "Count"],
    ["Total rows", len(df)],
    ["Unique metabolites", len(unique_names)],
    ["From cache", skipped],
    ["Newly fetched", len(unique_names) - skipped],
    ["", ""],
] + [[f"  {lbl}", filled[lbl]] for _, lbl in ann_output_cols]

for row in summary_rows:
    ws2.append(row)
ws2.cell(1,1).font = Font(bold=True, name="Arial")
ws2.cell(1,2).font = Font(bold=True, name="Arial")
ws2.column_dimensions["A"].width = 35
ws2.column_dimensions["B"].width = 12

wb.save(OUTPUT_FILE)
print(f"✅ Saved to {OUTPUT_FILE}")


✅ Saved to /content/drive/MyDrive/PD_metabolomics_annotated_new.xlsx


## 8 · Summary & download

In [ ]:
from IPython.display import HTML

rows = [
    ("Total rows processed", f"{len(df):,}"),
    ("Unique metabolites",   f"{len(unique_names):,}"),
    ("From cache",           f"{skipped:,}"),
    ("Newly annotated",      f"{len(unique_names)-skipped:,}"),
]
for _, lbl in ann_output_cols:
    pct = filled[lbl] / len(df) * 100
    rows.append((lbl, f"{filled[lbl]:,}  ({pct:.1f}%)"))

table = "<table style='border-collapse:collapse;font-family:Arial;font-size:13px'>"
table += "<tr><th style='background:#1F4E79;color:white;padding:6px 16px;text-align:left'>Metric</th>"
table += "<th style='background:#1F4E79;color:white;padding:6px 16px;text-align:right'>Value</th></tr>"
for i, (k, v) in enumerate(rows):
    bg = "#EBF3FB" if i % 2 == 0 else "white"
    if k == "":
        table += f"<tr><td colspan=2 style='padding:4px'></td></tr>"
    else:
        table += f"<tr style='background:{bg}'><td style='padding:5px 16px'>{k}</td><td style='padding:5px 16px;text-align:right'>{v}</td></tr>"
table += "</table>"
display(HTML(table))

# Download button
from google.colab import files
print(f"\n📥 Download your annotated file:")
files.download(OUTPUT_FILE)


Metric,Value
Total rows processed,20
Unique metabolites,20
From cache,4
Newly annotated,16
PubChem CID,20 (100.0%)
KEGG ID,16 (80.0%)
HMDB ID (filled),4 (20.0%)
Formula (filled),20 (100.0%)
InChIKey (filled),20 (100.0%)
SMILES (filled),4 (20.0%)



📥 Download your annotated file:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>